# Honeypot Overview — EDA

The "what did we collect" baseline (*Data-Driven Security*, ch. 3), shaped around how Cowrie and this deployment work. Exploration only — logic worth keeping graduates into `summary.py` / `charts.py`.

**Read these numbers with three Cowrie facts in mind** (details in `../README.md`):
- **Compare by `node_role`, not cloud or sensor.** AWS runs `control_proxy` (SSH only); Azure runs `agent_detector` (SSH + telnet + agent prompt handlers). `sensor` is a container id that changes on redeploy.
- **Authentication is config.** `userdb.txt` accepts `root`/`admin` with any password, so "authenticated" says which usernames attackers try, not how good they are. The attacker-driven signal is what happens *after* login.
- **`dst_port` is our own listening port** (2222/2223), except on tunnel requests.

**Setup**: the same "Python (bear-trap analysis)" kernel as `network/` and `fingerprinting/` — see `../README.md`.

In [ ]:
from IPython.display import Image

from honeypot.analysis.common.extract import get_sessions
from honeypot.analysis.overview.charts import (
    plot_daily_activity,
    plot_session_funnel,
    plot_tunnel_targets,
)
from honeypot.analysis.overview.summary import (
    coverage,
    daily_activity,
    session_funnel,
    tunnel_targets,
)

## Load data

`"offline"` reads the most recent local snapshot from `../common/outputs/snapshots/`; `"live"` pulls fresh from BigQuery and refreshes the snapshot.

In [ ]:
MODE = "offline"

events = get_sessions(MODE)
events.height

## Coverage: what does this dataset contain?

Read this before any other number: which roles and protocols reported, over what window, and how many sensor containers each went through.

In [ ]:
coverage(events)

## What sessions did

Share of each role/protocol's sessions that tried a login, authenticated, ran commands, transferred files, or asked the honeypot to tunnel traffic. Stages aren't strictly nested (a tunnel needs a login but no commands). Log scale: file transfers and tunnels sit orders of magnitude below logins.

In [ ]:
funnel = session_funnel(events)
funnel

In [ ]:
Image(plot_session_funnel(funnel))

## Who showed up, day by day

Distinct source IPs per UTC day per role, split into **new** (first seen anywhere in the dataset that day) vs **returning**. Counts IPs, not events, so a few chatty sessions can't dominate. The first and last (partial) collection days are trimmed; expect nearly everything to be "new" for the first few days while the series warms up.

In [ ]:
daily = daily_activity(events)
daily

In [ ]:
Image(plot_daily_activity(daily))

## Tunnel requests: using the honeypot as a relay

`cowrie.direct-tcpip.request` is an attacker asking the honeypot to open a connection onward. One target requested by hundreds of IPs (a mail server on port 25) reads as a relay-probing campaign; `ip-who.com` / `google.com` targets are usually connectivity checks.

In [ ]:
targets = tunnel_targets(events)
targets.head(15)

In [ ]:
Image(plot_tunnel_targets(targets))

## Exploration

Ad hoc from here. Open threads worth following up:
- What drove the agent_detector spikes (new-IP waves on 08-30 and 09-09, then a returning wave on 09-10/11)? Cross-reference `network/` clusters and, once enrichment lands, ASNs.
- Who are the port-25 relay probers — one campaign or many? (Ch. 4 enrichment.)
- Why does telnet authenticate a larger share of sessions than SSH, and run so much longer (median duration in `funnel`)?